# C4: Mask data before it reaches the model
Config `policy-chatbot`: PII is replaced by typed placeholders (`[NIK]`, `[PHONE]`, `[NAME]`...) **before** the prompt is sent to Qwen3-8B.
We show what the model actually receives, then score 30 answers for usefulness (LLM judge, temperature 0; review the flagged ones by hand).

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
t = h.transform("My NIK is 3171041208850123 and my phone is +62 812-3456-7890. How do I open a savings account?", config_id="policy-chatbot")
print("model receives ->", t["text_out"])

**Reading the output:** this is the exact text the model receives. The NIK and phone number were replaced with typed placeholders (`[NIK]`, `[PHONE]`), so the model never sees the real values but still knows what kind of information the customer gave, and can answer the question.

In [ ]:
import c4_eval
rows = c4_eval.main()

**Reading the output** (30 realistic customer prompts, each masked, answered by the model, then judged):

* `PII values masked before the model: X/Y`: of all personal-data values in the prompts, how many were replaced before reaching the model. Target **100%** (41/41 in the reference run).
* `Answers judged useful: X/30`: a separate judge call checks whether the answer still covers the expected points despite the masking (28/30 in the reference run). Masking shouldn't make the bot useless.
* `Answers repeating original PII: 0`: no answer contains a real value, because the model never had it.
* Lines starting with `CHECK` are samples to review by hand: an answer judged not useful, or a prompt where a value was not masked. The judge is a model too, so read them rather than trusting the score blindly.

In [ ]:
import pandas as pd
pd.DataFrame(rows)[["id","model_received","useful","answer_repeats_pii"]].head(30)

**Columns:** `model_received` is the masked prompt the model saw; `useful` is the judge's verdict; `answer_repeats_pii` is `True` only if an answer contained an original value (it should be `False` everywhere).